In [ ]:
from shapely.geometry import Point
from rasterstats import zonal_stats
import geopandas as gpd

In [ ]:
WATER_DATA = "output/GHSL_LAND_E2018_GLOBE_90_merged.tif"
WATER_CHECK_TEST_POINT = Point(452176.797, 537829.255)
WATER_DISTANCE_TEST_POINTS = "test_points/test_points.shp"
TOLERANCE_M = 10

In [ ]:
def check_for_water(point: Point, radius_m: float, water_map: str = WATER_DATA, water_pixel_value: float = 1.0) -> bool:
    """
    Check whether there is water near a given point within a search radius.

    Expects:
        point: shapely.Point object with the location to check (reference system need to match water map).
        radius_m: Search radius in meters.
        water_map: Path to GeoTIF file with water map.
        water_pixel_value: Value of pixels with water.

    Returns:
        True if water is found within `radius_m` of `point`, otherwise False.
    """
    stats = zonal_stats(
        point.buffer(radius_m),
        water_map,
        stats="max",
    )
    if stats[0]["max"] == water_pixel_value:
        return True
    else:
        return False

In [ ]:
assert check_for_water(WATER_CHECK_TEST_POINT, 930) == True, "Should return True"
assert check_for_water(WATER_CHECK_TEST_POINT, 920) == False, "Should return False"

In [ ]:
def find_water_distance(point, tolerance=TOLERANCE_M):
    """
    Find distance to water for given location.

    Expects:
        point: shapely.Point object with the location to check (reference system need to match water map).
        tolerance: Maximum error of distance.

    Returns:
        Integer value of distance to water in meters.
    """
    radius = tolerance
    water_found = check_for_water(point, radius)
    
    while not water_found:
        radius *= 10
        water_found = check_for_water(point, radius)
    
    min_val = 0
    max_val = radius
    candidate_val = round((min_val + max_val) / 2)
    difference = max_val - min_val

    while difference > tolerance:
        water_found = check_for_water(point, candidate_val)
        if water_found:
            max_val = candidate_val
        else:
            min_val = candidate_val
        difference = max_val - min_val
        candidate_val = round((min_val + max_val) / 2)

    return candidate_val

In [ ]:
find_water_distance(TEST_POINT)